In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [3]:
#Load the dataset
trans_df=pd.read_csv('../data/transactions_cleaned.csv')
trans_df.head()

,trans_id,trans_datetime,cc_num,merchant,category,amt,firstname,lastname,gender,street,...,trans_day,trans_hour,time_period,fullname,high_amount_risk,night_risk,txn_per_card,freq_risk,location_risk,risk_score
0,1,2020-06-21 12:14:00,2.291160e+15,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,Sunday,12,Day,Jeff Elliott,0,0,640,1,0,1
1,2,2020-06-21 12:14:00,3.573030e+15,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,Sunday,12,Day,Joanne Williams,0,0,837,1,0,1
2,3,2020-06-21 12:14:00,3.598220e+15,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,Sunday,12,Day,Ashley Lopez,0,0,1073,1,0,1
3,4,2020-06-21 12:15:00,3.591920e+15,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,Sunday,12,Day,Brian Williams,0,0,663,1,0,1
4,5,2020-06-21 12:15:00,3.526830e+15,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,Sunday,12,Day,Nathan Massey,0,0,891,1,0,1


In [5]:
X = trans_df[['high_amount_risk','night_risk','freq_risk','location_risk','risk_score']]
y = trans_df['is_fraud']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [10]:
y_prob = model.predict_proba(X_test)[:, 1]

In [11]:
df_test = X_test.copy()
df_test["actual"] = y_test
df_test["fraud_probability"] = y_prob


In [12]:
df_test["risk_score_ml"] = df_test["fraud_probability"] * 100

In [13]:
df_test["final_risk_score"] = (
    df_test["risk_score_ml"] +
    df_test["high_amount_risk"] * 10 +
    df_test["night_risk"] * 5 +
    df_test["freq_risk"] * 10 +
    df_test["location_risk"] * 8
)

In [14]:
def risk_level(score):
    if score > 80:
        return "High Risk"
    elif score > 40:
        return "Medium Risk"
    else:
        return "Low Risk"

df_test["risk_level"] = df_test["final_risk_score"].apply(risk_level)

In [15]:
def action(level):
    if level == "High Risk":
        return "Block Transaction"
    elif level == "Medium Risk":
        return "Manual Review"
    else:
        return "Approve"

df_test["action"] = df_test["risk_level"].apply(action)

In [16]:
df_test.head()

,high_amount_risk,night_risk,freq_risk,location_risk,risk_score,actual,fraud_probability,risk_score_ml,final_risk_score,risk_level,action
119106,0,0,1,0,1,0,0.173038,17.303799,27.303799,Low Risk,Approve
179292,0,0,1,0,1,0,0.173038,17.303799,27.303799,Low Risk,Approve
540729,0,0,1,0,1,0,0.173038,17.303799,27.303799,Low Risk,Approve
374360,0,0,1,0,1,0,0.173038,17.303799,27.303799,Low Risk,Approve
314574,0,1,1,0,2,0,0.293512,29.351213,44.351213,Medium Risk,Manual Review


In [17]:
df_test.to_csv("../outputs/final_fraud_risk_analysis.csv", index=False)